# NB06 · Explicabilidad visual con Grad-CAM

**Entorno recomendado:** CPU, sin acelerador  
**GPU:** opcional; se utiliza automáticamente si está disponible  
**Tiempo estimado:** CPU, 10–15 min; T4/L4, 5–8 min

Este notebook explica visualmente las predicciones del modelo final `YOLO11m · semilla 2026`. Para cada fotografía genera cuatro paneles:

1. fotografía original;
2. recorte cuadrado realmente analizado por el modelo;
3. mapa Grad-CAM;
4. superposición del mapa sobre la entrada del modelo.

La fotografía original conserva su relación de aspecto. El recorte cuadrado se muestra por separado para que el lector comprenda qué información recibió la red neuronal.

> Grad-CAM identifica regiones asociadas con una predicción; no demuestra causalidad ni sustituye la interpretación agronómica.


In [ ]:
%pip install -q -U ultralytics grad-cam


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import platform
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ultralytics
from PIL import Image, ImageOps
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
MODEL_PATH = PROJECT_ROOT / '05_MODELOS' / 'yolo11m_cls_seed2026' / 'best.pt'
NB05_RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS' / 'NB05_DEMOSTRADOR_INFERENCIA'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS' / 'NB06_EXPLICABILIDAD_GRADCAM'
BOOK_ROOT = PROJECT_ROOT / '08_MATERIAL_LIBRO' / 'NB06_EXPLICABILIDAD_GRADCAM'
LOCAL_INPUT = Path('/content/palma_gradcam_input')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_RESULTS = RESULTS_ROOT / RUN_ID
RUN_BOOK = BOOK_ROOT / RUN_ID
ORIGINALS_DIR = RUN_RESULTS / 'imagenes_originales'
FIGURES_DIR = RUN_BOOK / 'figuras_gradcam'

for path in [RUN_RESULTS, RUN_BOOK, ORIGINALS_DIR, FIGURES_DIR, LOCAL_INPUT]:
    path.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'No se encontró el modelo final: {MODEL_PATH}. Ejecute NB04.')

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
IMGSZ = 384
CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SPANISH = {
    'damaged': 'dañado',
    'empty': 'vacío',
    'overripe': 'sobremaduro',
    'ripe': 'maduro',
    'unripe': 'inmaduro',
}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

print('Dispositivo:', DEVICE_NAME)
print('Modelo:', MODEL_PATH)
print('Ultralytics:', ultralytics.__version__)
print('Resultados:', RUN_RESULTS)
print('Material para el libro:', RUN_BOOK)


## Selección de imágenes

Por defecto se reutilizan automáticamente las imágenes de la ejecución más reciente de NB05. Si desea analizar otras fotografías, cambie `USE_LATEST_NB05_IMAGES` a `False`; aparecerá el selector de archivos.


In [ ]:
USE_LATEST_NB05_IMAGES = True

def valid_images(folder):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


selected_paths = []
if USE_LATEST_NB05_IMAGES and NB05_RESULTS_ROOT.exists():
    previous_runs = sorted(
        [p for p in NB05_RESULTS_ROOT.iterdir() if p.is_dir()],
        reverse=True,
    )
    for previous_run in previous_runs:
        candidates = valid_images(previous_run / 'imagenes_originales')
        if candidates:
            for source in candidates:
                destination = LOCAL_INPUT / source.name
                shutil.copy2(source, destination)
                selected_paths.append(destination)
            print('Imágenes reutilizadas desde:', previous_run)
            break

if not selected_paths:
    from google.colab import files
    uploaded = files.upload()
    for filename, data in uploaded.items():
        destination = LOCAL_INPUT / Path(filename).name
        destination.write_bytes(data)
        if destination.suffix.lower() in IMAGE_EXTS:
            try:
                with Image.open(destination) as im:
                    im.verify()
                selected_paths.append(destination)
            except Exception as exc:
                print(f'Omitida {destination.name}: {exc}')

selected_paths = sorted(set(selected_paths))
if not selected_paths:
    raise RuntimeError('No se encontraron imágenes válidas para explicar.')

for source in selected_paths:
    shutil.copy2(source, ORIGINALS_DIR / source.name)

print('Imágenes seleccionadas:', len(selected_paths))
for path in selected_paths:
    print('-', path.name)


## Preparación del modelo y selección de capa

Se utiliza la salida convolucional del cabezal de clasificación, antes del agrupamiento global. Esta capa mantiene información espacial y es apropiada para construir el mapa Grad-CAM.

El preprocesamiento replica la inferencia de Ultralytics: redimensionamiento, recorte central, conversión a tensor y normalización.


In [ ]:
yolo = YOLO(str(MODEL_PATH))
base_model = yolo.model.to(DEVICE).eval()

# Los pesos cargados para inferencia pueden quedar congelados. Grad-CAM necesita
# construir el grafo de autograd, aunque el modelo permanezca en modo evaluación.
for parameter in base_model.parameters():
    parameter.requires_grad_(True)

model_names = [base_model.names[i] for i in sorted(base_model.names)]
if model_names != CLASSES:
    raise RuntimeError(f'Clases inesperadas: {model_names}')

classification_head = base_model.model[-1]
if not hasattr(classification_head, 'conv'):
    raise RuntimeError('No se encontró la capa convolucional del cabezal de clasificación.')
target_layer = classification_head.conv


class LogitsWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output = self.model(x)
        if isinstance(output, (tuple, list)):
            # En evaluación, el clasificador devuelve (probabilidades, logits).
            if len(output) > 1 and torch.is_tensor(output[1]):
                return output[1]
            output = output[0]
        if not torch.is_tensor(output):
            raise TypeError(f'Salida del modelo no compatible: {type(output)}')
        return output


cam_model = LogitsWrapper(base_model).to(DEVICE).eval()
transform = classify_transforms(IMGSZ)
geometry_transform = type(transform)(transform.transforms[:2])
cam = GradCAM(model=cam_model, target_layers=[target_layer])

print('Capa objetivo:', target_layer.__class__.__name__)
print('Transformaciones:', [t.__class__.__name__ for t in transform.transforms])


In [ ]:
def center_of_mass(cam_map):
    weights = np.clip(cam_map.astype(float), 0, None)
    total = weights.sum()
    if total <= 1e-12:
        return 0.5, 0.5
    yy, xx = np.indices(weights.shape)
    return float((xx * weights).sum() / total / max(weights.shape[1] - 1, 1)), float((yy * weights).sum() / total / max(weights.shape[0] - 1, 1))


records = []
start_all = time.perf_counter()

for image_path in selected_paths:
    with Image.open(image_path) as im:
        original = ImageOps.exif_transpose(im).convert('RGB')

    model_view_pil = geometry_transform(original)
    model_view_rgb = np.asarray(model_view_pil, dtype=np.float32) / 255.0
    input_tensor = transform(original).unsqueeze(0).to(DEVICE).requires_grad_(True)

    with torch.no_grad():
        logits = cam_model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]
        predicted_id = int(probabilities.argmax().item())
        predicted_score = float(probabilities[predicted_id].item())

    target = [ClassifierOutputTarget(predicted_id)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=target)[0]
    grayscale_cam = np.clip(grayscale_cam, 0, 1)
    overlay = show_cam_on_image(model_view_rgb, grayscale_cam, use_rgb=True, image_weight=0.58)

    predicted_class = CLASSES[predicted_id]
    com_x, com_y = center_of_mass(grayscale_cam)
    record = {
        'imagen': image_path.name,
        'clase_predicha': predicted_class,
        'clase_predicha_es': SPANISH[predicted_class],
        'puntaje': predicted_score,
        'capa_objetivo': target_layer.__class__.__name__,
        'centro_activacion_x_norm': com_x,
        'centro_activacion_y_norm': com_y,
        'activacion_media': float(grayscale_cam.mean()),
        'activacion_maxima': float(grayscale_cam.max()),
    }
    for class_id, class_name in enumerate(CLASSES):
        record[f'puntaje_{class_name}'] = float(probabilities[class_id].item())
    records.append(record)

    # Composición 2 × 2: conserva el tamaño de los rótulos y de los detalles
    # cuando la figura se inserta en una página vertical del libro.
    fig, axes_grid = plt.subplots(2, 2, figsize=(10.2, 8.8), constrained_layout=True)
    axes = axes_grid.ravel()
    fig.patch.set_facecolor('white')

    axes[0].imshow(original, aspect='equal')
    axes[0].set_title('a) Fotografía original', fontweight='semibold')
    axes[0].axis('off')

    axes[1].imshow(model_view_rgb, aspect='equal')
    axes[1].set_title(f'b) Entrada del modelo\n{IMGSZ} × {IMGSZ} píxeles', fontweight='semibold')
    axes[1].axis('off')

    heat = axes[2].imshow(grayscale_cam, cmap='jet', vmin=0, vmax=1, aspect='equal')
    axes[2].set_title('c) Mapa Grad-CAM', fontweight='semibold')
    axes[2].axis('off')

    axes[3].imshow(overlay, aspect='equal')
    axes[3].set_title('d) Superposición', fontweight='semibold')
    axes[3].axis('off')

    fig.colorbar(heat, ax=axes[2], fraction=0.046, pad=0.03, label='Activación relativa')
    fig.suptitle(
        f"Explicación visual · Predicción: {SPANISH[predicted_class].upper()} "
        f"(puntaje {predicted_score:.3f})",
        fontsize=15, fontweight='bold'
    )
    output_path = FIGURES_DIR / f'{image_path.stem}_gradcam.png'
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches='tight',
        pad_inches=0.08,
        facecolor='white',
    )
    plt.show()

elapsed = time.perf_counter() - start_all
explanations = pd.DataFrame(records)
explanations.to_csv(RUN_RESULTS / 'resumen_gradcam.csv', index=False)
display(explanations.round(4))
print(f'Tiempo total: {elapsed:.2f} s')


In [ ]:
# Montaje compacto para el libro: superposiciones de hasta 12 imágenes.
n_show = min(12, len(selected_paths))

# Con una sola fotografía se genera una lámina individual, sin el lienzo vacío
# propio de una cuadrícula. Con dos o más se construye el montaje comparativo.
if n_show == 1:
    fig, ax = plt.subplots(figsize=(5.4, 5.9), constrained_layout=True)
    axes = np.asarray([[ax]])
else:
    ncols = min(3, n_show)
    nrows = math.ceil(n_show / ncols)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4*ncols, 4*nrows),
        squeeze=False,
        constrained_layout=True,
    )

fig.patch.set_facecolor('white')
for ax in axes.flat:
    ax.axis('off')

for ax, image_path, (_, row) in zip(
    axes.flat,
    selected_paths[:n_show],
    explanations.head(n_show).iterrows(),
):
    figure_path = FIGURES_DIR / f'{image_path.stem}_gradcam.png'
    # Para el montaje se recalcula únicamente la superposición, evitando recortar la figura completa.
    with Image.open(image_path) as im:
        original = ImageOps.exif_transpose(im).convert('RGB')
    model_view_pil = geometry_transform(original)
    model_view_rgb = np.asarray(model_view_pil, dtype=np.float32) / 255.0
    input_tensor = transform(original).unsqueeze(0).to(DEVICE).requires_grad_(True)
    predicted_id = CLASSES.index(row['clase_predicha'])
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[ClassifierOutputTarget(predicted_id)],
    )[0]
    overlay = show_cam_on_image(model_view_rgb, np.clip(grayscale_cam, 0, 1), use_rgb=True, image_weight=0.58)
    ax.imshow(overlay, aspect='equal')
    ax.set_title(
        f"{row['clase_predicha_es'].upper()} · {row['puntaje']:.3f}",
        fontsize=10, fontweight='bold'
    )
    ax.axis('off')

fig.suptitle(
    'Regiones asociadas con la clasificación · Grad-CAM',
    fontsize=14,
    fontweight='bold',
)
montage_path = RUN_BOOK / 'montaje_gradcam.png'
fig.savefig(
    montage_path,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0.08,
    facecolor='white',
)
plt.show()
print('Montaje guardado:', montage_path)


In [ ]:
metadata = {
    'run_id': RUN_ID,
    'method': 'Grad-CAM',
    'model': 'YOLO11m-cls',
    'seed': 2026,
    'image_size': IMGSZ,
    'target_layer': str(target_layer.__class__.__name__),
    'n_images': len(selected_paths),
    'device': DEVICE_NAME,
    'elapsed_seconds': round(elapsed, 4),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'interpretation_warning': 'Grad-CAM muestra asociación espacial; no demuestra causalidad.',
}
(RUN_RESULTS / 'metadatos_gradcam.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

cam.activations_and_grads.release()

print('NB06 FINALIZADO CORRECTAMENTE')
print('Resultados:', RUN_RESULTS)
print('Figuras para el libro:', RUN_BOOK)


## Lectura del mapa

- Los colores cálidos indican regiones con mayor asociación con la clase predicha.
- Los colores fríos indican menor activación relativa.
- Un mapa concentrado sobre el racimo es más coherente que uno dominado por el cielo, el tronco o el fondo.
- La comparación entre la fotografía original y el recorte de entrada permite detectar información eliminada por el preprocesamiento.
- Estas visualizaciones deben interpretarse junto con las métricas y las limitaciones del conjunto de datos.
